In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from create_window_util import create_windows

# LSTM

In [7]:
FILE_PATH = "../processed_data/SDHAR/final_processed_data_ALL_DAYS_small_window.csv"
TARGET_COLUMN = 'activity_user_1'  # We will predict the activity for User 1
WINDOW_SIZE = 60  # How many past time steps to look at (e.g., 60 * 1s = 60 seconds of history)
STEP_SIZE = 30    # How far to slide the window forward each time

print("Loading final processed data...")
df = pd.read_csv(FILE_PATH)

print("Separating data...")
df.dropna(inplace=True)
X = df.drop(columns=[col for col in df.columns if 'activity' in col])
y = df[TARGET_COLUMN].astype(int)
num_classes = len(y.unique())
y_categorical = to_categorical(y, num_classes=num_classes)
print(f"Creating sliding windows (size={WINDOW_SIZE}, step={STEP_SIZE})...")
X_win, y_win = create_windows(X, pd.Series(y_categorical.tolist()), WINDOW_SIZE, STEP_SIZE)
print(f"  - Windowed X shape: {X_win.shape}")
print(f"  - Windowed y shape: {y_win.shape}")

print("Splitting data into training and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X_win, y_win, test_size=0.2, random_state=42)
print(f"  - Training set size: {len(X_train)}")
print(f"  - Test set size: {len(X_test)}")

print("Building the LSTM model...")
model = Sequential([
    # The input layer must match the shape of our windows (WINDOW_SIZE, num_features)
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True),
    Dropout(0.5),
    LSTM(64),
    Dropout(0.5),
    Dense(num_classes, activation='softmax') # The output layer has one neuron per activity
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

print("\nTraining the model...")
history = model.fit(
    X_train, y_train,
    epochs=10,          # Start with a few epochs to see how it goes
    batch_size=128,
    validation_split=0.1, # Use part of the training data for validation
    verbose=1
)
model.save('../models/SDHAR/LSTM_small_window.keras')

print("\nEvaluating the model on the test set...")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

print("\nClassification Report:")
print(classification_report(y_test_labels, y_pred))

Loading final processed data...
Separating data...
Creating sliding windows (size=60, step=30)...
  - Windowed X shape: (134934, 60, 41)
  - Windowed y shape: (134934, 18)
Splitting data into training and test sets...
  - Training set size: 107947
  - Test set size: 26987
Building the LSTM model...


C:\Users\jesse\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 64)         │        27,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 18)             │         1,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,330 (239.57 KB)

 Trainable params: 61,330 (239.57 KB)

 Non-trainable params: 0 (0.00 B)


Training the model...
Epoch 1/10
759/759 ━━━━━━━━━━━━━━━━━━━━ 582s 728ms/step - accuracy: 0.8276 - loss: 0.6126 - val_accuracy: 0.8727 - val_loss: 0.3975
Epoch 2/10
759/759 ━━━━━━━━━━━━━━━━━━━━ 59s 78ms/step - accuracy: 0.8853 - loss: 0.3720 - val_accuracy: 0.8917 - val_loss: 0.3358
Epoch 3/10
759/759 ━━━━━━━━━━━━━━━━━━━━ 60s 80ms/step - accuracy: 0.9024 - loss: 0.3169 - val_accuracy: 0.9109 - val_loss: 0.2821
Epoch 4/10
759/759 ━━━━━━━━━━━━━━━━━━━━ 62s 81ms/step - accuracy: 0.9137 - loss: 0.2805 - val_accuracy: 0.9217 - val_loss: 0.2486
Epoch 5/10
759/759 ━━━━━━━━━━━━━━━━━━━━ 64s 84ms/step - accuracy: 0.9211 - loss: 0.2584 - val_accuracy: 0.9226 - val_loss: 0.2444
Epoch 6/10
759/759 ━━━━━━━━━━━━━━━━━━━━ 66s 87ms/step - accuracy: 0.9245 - loss: 0.2434 - val_accuracy: 0.9302 - val_loss: 0.2200
Epoch 7/10
759/759 ━━━━━━━━━━━━━━━━━━━━ 68s 89ms/step - accuracy: 0.9313 - loss: 0.2232 - val_accuracy: 0.9326 - val_loss: 0.2099
Epoch 8/10
759/759 ━━━━━━━━━━━━━━━━━━━━ 68s 89ms/step - accuracy:

C:\Users\jesse\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\jesse\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\jesse\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

# Random Forest

In [3]:
FILE_PATH = "../processed_data/SDHAR/final_processed_data_ALL_DAYS_small_window.csv"
TARGET_COLUMN = 'activity_user_1'
WINDOW_SIZE = 60  # 120 seconds of history
STEP_SIZE = 30

print("Loading final processed data...")
df = pd.read_csv(FILE_PATH)

print("Separating Features and Target...")
df.dropna(inplace=True)
X = df.drop(columns=[col for col in df.columns if 'activity' in col])
y = df[TARGET_COLUMN].astype(int)

print(f"Creating sliding windows (size={WINDOW_SIZE}, step={STEP_SIZE})...")
X_win, y_win = create_windows(X, y, WINDOW_SIZE, STEP_SIZE)
print(f"  - Initial windowed X shape: {X_win.shape}")

print("Flattening window data...")
n_samples, n_timesteps, n_features = X_win.shape
X_flattened = X_win.reshape((n_samples, n_timesteps * n_features))
print(f"  - Flattened X shape: {X_flattened.shape}")

print("Splitting data into training and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X_flattened, y_win, test_size=0.2, random_state=42)
print(f"  - Training set size: {len(X_train)}")
print(f"  - Test set size: {len(X_test)}")

print("\nBuilding and training the Random Forest model...")
# n_estimators is the number of trees in the forest.
# n_jobs=-1 uses all available CPU cores for faster training.
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

model.fit(X_train, y_train)

print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test)
joblib.dump(model, "../models/SDHAR/RandomForest_small_window.joblib")

accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
# You may need to create a mapping from integer back to activity name for readability
activity_names = ["BATHROOM ACTIVITY", "CHORES", "COOK", "DISHWASHING", "DRESS", "EAT", "LAUNDRY",
                  "MAKE SIMPLE FOOD", "OUT HOME", "PET", "READ", "RELAX", "SHOWER", "SLEEP",
                  "TAKE MEDS", "WATCH TV", "WORK", "OTHER"]
print(classification_report(y_test, y_pred, target_names=activity_names))

Loading final processed data...
Separating Features and Target...
Creating sliding windows (size=60, step=30)...
  - Initial windowed X shape: (134934, 60, 41)
Flattening window data...
  - Flattened X shape: (134934, 2460)
Splitting data into training and test sets...
  - Training set size: 107947
  - Test set size: 26987

Building and training the Random Forest model...

Evaluating the model on the test set...

Test Accuracy: 98.67%

Classification Report:
                   precision    recall  f1-score   support

BATHROOM ACTIVITY       0.93      0.95      0.94       792
           CHORES       0.96      0.89      0.93       139
             COOK       0.97      0.94      0.95       266
      DISHWASHING       0.97      1.00      0.99        33
            DRESS       0.90      0.51      0.65        69
              EAT       0.98      0.97      0.97      1257
          LAUNDRY       1.00      0.40      0.57         5
 MAKE SIMPLE FOOD       0.91      0.86      0.89       167
     

# Decision Tree

In [6]:
FILE_PATH = "../processed_data/SDHAR/final_processed_data_ALL_DAYS_small_window.csv"
TARGET_COLUMN = 'activity_user_1'
WINDOW_SIZE = 60
STEP_SIZE = 30

print("Loading final processed data...")
df = pd.read_csv(FILE_PATH)

print("Separating Features and Target...")
df.dropna(inplace=True)
X = df.drop(columns=[col for col in df.columns if 'activity' in col])
y = df[TARGET_COLUMN].astype(int)

print(f"Creating sliding windows (size={WINDOW_SIZE}, step={STEP_SIZE})...")
X_win, y_win = create_windows(X, y, WINDOW_SIZE, STEP_SIZE)
print(f"  - Initial windowed X shape: {X_win.shape}")

print("Flattening window data...")
n_samples, n_timesteps, n_features = X_win.shape
X_flattened = X_win.reshape((n_samples, n_timesteps * n_features))
print(f"  - Flattened X shape: {X_flattened.shape}")

print("Splitting data into training and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X_flattened, y_win, test_size=0.2, random_state=42)
print(f"  - Training set size: {len(X_train)}")
print(f"  - Test set size: {len(X_test)}")

print("Building and training the Decision Tree model...")
model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)
print("  - Model training complete!")

print("Evaluating the model on the test set...")
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
activity_names = ["BATHROOM ACTIVITY", "CHORES", "COOK", "DISHWASHING", "DRESS", "EAT", "LAUNDRY",
                  "MAKE SIMPLE FOOD", "OUT HOME", "PET", "READ", "RELAX", "SHOWER", "SLEEP",
                  "TAKE MEDS", "WATCH TV", "WORK", "OTHER"]
print(classification_report(y_test, y_pred, target_names=activity_names))

print("Saving the Decision Tree model...")
joblib.dump(model, "../models/SDHAR/DecisionTree_small_window.joblib")
print("  - Model saved successfully!")

Loading final processed data...
Separating Features and Target...
Creating sliding windows (size=60, step=30)...
  - Initial windowed X shape: (134934, 60, 41)
Flattening window data...
  - Flattened X shape: (134934, 2460)
Splitting data into training and test sets...
  - Training set size: 107947
  - Test set size: 26987
Building and training the Decision Tree model...
  - Model training complete!
Evaluating the model on the test set...

Test Accuracy: 97.14%

Classification Report:
                   precision    recall  f1-score   support

BATHROOM ACTIVITY       0.93      0.93      0.93       792
           CHORES       0.76      0.81      0.78       139
             COOK       0.88      0.87      0.87       266
      DISHWASHING       0.81      0.91      0.86        33
            DRESS       0.46      0.46      0.46        69
              EAT       0.93      0.93      0.93      1257
          LAUNDRY       1.00      0.60      0.75         5
 MAKE SIMPLE FOOD       0.70      0.8